# Aula 4 - Mastering Machine Learning Advanced

## NLP Aplicado: do TF-IDF ao BERT

### Prof. Dr. Ahirton Lopes (profahirton.lopes@fiap.com.br)

Ref. SMS Spam Collection Dataset — https://archive.ics.uci.edu/ml/datasets/sms+spam+collection

## Índice

1. [Introdução](#1)
2. [Carregamento e Exploração dos Dados](#2)
3. [Pré-processamento de Texto](#3)
   - 3.1 [Tokenização e Limpeza](#31)
   - 3.2 [Stopwords e Stemming](#32)
   - 3.3 [Análise de Frequência](#33)
4. [Abordagem Clássica: TF-IDF + ML](#4)
5. [Word Embeddings](#5)
   - 5.1 [Embeddings com Keras](#51)
   - 5.2 [LSTM para Classificação de Texto](#52)
6. [BERT com HuggingFace](#6)
7. [Comparativo Final](#7)
8. [Conclusão](#8)

# 1. Introdução <a id="1"></a>

**NLP (Natural Language Processing)** é o campo de ML voltado para ensinar computadores a entender, interpretar e gerar linguagem humana. Com a explosão de dados textuais — mensagens, avaliações, tickets de suporte, contratos — NLP tornou-se uma das habilidades mais demandadas em projetos de dados.

**Contexto de negócio:** operadoras de telecomunicações processam bilhões de SMS mensalmente. Identificar automaticamente mensagens de spam protege os clientes e reduz custo operacional. Vamos construir um detector de SMS spam do zero, evoluindo do mais simples ao mais poderoso:

![NLP Pipeline](https://i.imgur.com/OJtmQKX.png)

| Abordagem | Representação do texto | Complexidade |
|---|---|---|
| TF-IDF + Naive Bayes / LR | Bag of Words ponderado | Baixa |
| Embedding + LSTM | Vetores densos + memória sequencial | Média |
| BERT | Contexto bidirecional pré-treinado | Alta |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

np.random.seed(42)

# baixando recursos do NLTK
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# 2. Carregamento e Exploração dos Dados <a id="2"></a>

O **SMS Spam Collection Dataset** contém 5.574 mensagens de SMS rotuladas como `ham` (legítimas) ou `spam`. É um benchmark clássico para classificação de texto — e um caso de uso real de qualquer operadora de telecomunicações.

In [ ]:
# carregando o dataset
url = 'https://raw.githubusercontent.com/ahirtonlopes/Security-ML/main/Aula3_Extras/spam_data.csv'
df = pd.read_csv(url)
df.columns = ['label', 'message']
df.head(10)

In [ ]:
print(f'Total de mensagens: {len(df)}')
print(f'\nDistribuição de classes:')
print(df['label'].value_counts())

# convertendo label para numérico
df['label_num'] = (df['label'] == 'spam').astype(int)

# distribuição
df['label'].value_counts().plot(
    kind='bar', color=['steelblue', 'coral'], edgecolor='white', figsize=(6, 4)
)
plt.title('Distribuição de Classes — SMS Spam Dataset')
plt.xlabel('Classe')
plt.ylabel('Quantidade')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# analisando o comprimento das mensagens
df['num_chars']  = df['message'].apply(len)
df['num_words']  = df['message'].apply(lambda x: len(x.split()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for label, color in [('ham', 'steelblue'), ('spam', 'coral')]:
    subset = df[df['label'] == label]
    axes[0].hist(subset['num_chars'], bins=40, alpha=0.6, color=color, label=label, edgecolor='white')
    axes[1].hist(subset['num_words'], bins=30, alpha=0.6, color=color, label=label, edgecolor='white')

axes[0].set_title('Distribuição: Número de Caracteres')
axes[0].set_xlabel('Caracteres')
axes[0].legend()
axes[1].set_title('Distribuição: Número de Palavras')
axes[1].set_xlabel('Palavras')
axes[1].legend()

plt.suptitle('Spam vs Ham — Características das Mensagens', fontsize=13)
plt.tight_layout()
plt.show()

print(df.groupby('label')[['num_chars', 'num_words']].mean().round(1))

Já podemos notar que mensagens de spam tendem a ser mais longas. Isso sugere que features simples como comprimento do texto podem ter poder preditivo — mas vamos além disso.

# 3. Pré-processamento de Texto <a id="3"></a>

Antes de qualquer modelo, o texto precisa ser limpo e padronizado. Ruído como pontuação, maiúsculas e stopwords aumentam a dimensionalidade sem agregar informação.

## 3.1 Tokenização e Limpeza <a id="31"></a>

In [ ]:
def limpar_texto(texto):
    # lowercase
    texto = texto.lower()
    # removendo URLs
    texto = re.sub(r'http\S+|www\S+', '', texto)
    # removendo números
    texto = re.sub(r'\d+', '', texto)
    # removendo pontuação
    texto = texto.translate(str.maketrans('', '', string.punctuation))
    # removendo espaços extras
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# exemplo
exemplo = df['message'].iloc[1]
print('Original:  ', exemplo)
print('Limpo:     ', limpar_texto(exemplo))

In [ ]:
df['message_clean'] = df['message'].apply(limpar_texto)
df[['message', 'message_clean']].head(5)

## 3.2 Stopwords e Stemming <a id="32"></a>

- **Stopwords**: palavras muito frequentes que carregam pouca informação semântica ("the", "is", "at")
- **Stemming**: reduz as palavras à sua raiz ("running" → "run", "calls" → "call")

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer    = PorterStemmer()

def preprocessar(texto):
    tokens = word_tokenize(texto)
    tokens = [stemmer.stem(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['message_proc'] = df['message_clean'].apply(preprocessar)

print('Limpo:       ', df['message_clean'].iloc[1])
print('Processado:  ', df['message_proc'].iloc[1])

## 3.3 Análise de Frequência <a id="33"></a>

In [ ]:
def palavras_mais_comuns(df, label, n=20):
    textos = ' '.join(df[df['label'] == label]['message_proc'])
    return Counter(textos.split()).most_common(n)

palavras_spam = palavras_mais_comuns(df, 'spam')
palavras_ham  = palavras_mais_comuns(df, 'ham')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, palavras, titulo, cor in [
    (axes[0], palavras_spam, 'SPAM — Palavras mais frequentes', 'coral'),
    (axes[1], palavras_ham,  'HAM  — Palavras mais frequentes', 'steelblue')
]:
    words, counts = zip(*palavras)
    ax.barh(list(reversed(words)), list(reversed(counts)), color=cor, edgecolor='white')
    ax.set_title(titulo)
    ax.set_xlabel('Frequência')

plt.tight_layout()
plt.show()

# 4. Abordagem Clássica: TF-IDF + ML <a id="4"></a>

**TF-IDF (Term Frequency - Inverse Document Frequency)** transforma cada mensagem em um vetor numérico, onde o peso de cada palavra reflete sua importância relativa no documento e sua raridade no corpus.

- **TF**: frequência da palavra no documento
- **IDF**: penaliza palavras muito comuns em todos os documentos

Combinamos com três classificadores clássicos que funcionam bem com dados esparsos de texto.

In [ ]:
X = df['message_proc']
y = df['label_num']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# testando três classificadores com TF-IDF
classificadores = {
    'Naive Bayes':         MultinomialNB(),
    'Regressão Logística': LogisticRegression(max_iter=500),
    'SVM Linear':          LinearSVC()
}

resultados_tfidf = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nome, clf in classificadores.items():
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
        ('clf',   clf)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    resultados_tfidf.append({
        'Modelo': nome,
        'Acurácia': accuracy_score(y_test, y_pred),
        'Precision': float(classification_report(y_test, y_pred, output_dict=True)['1']['precision']),
        'Recall':    float(classification_report(y_test, y_pred, output_dict=True)['1']['recall']),
        'F1-Score':  float(classification_report(y_test, y_pred, output_dict=True)['1']['f1-score']),
    })

df_tfidf = pd.DataFrame(resultados_tfidf).set_index('Modelo').round(4)
df_tfidf

In [ ]:
# matriz de confusão do melhor modelo TF-IDF (SVM)
pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf',   LinearSVC())
])
pipeline_svm.fit(X_train, y_train)
y_pred_svm = pipeline_svm.predict(X_test)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_svm,
    display_labels=['Ham', 'Spam'],
    cmap='Blues', ax=ax
)
plt.title('Matriz de Confusão — TF-IDF + SVM Linear')
plt.tight_layout()
plt.show()

In [ ]:
# analisando as features mais importantes para spam vs ham
pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
    ('clf',   LogisticRegression(max_iter=500))
])
pipeline_lr.fit(X_train, y_train)

feature_names = pipeline_lr.named_steps['tfidf'].get_feature_names_out()
coefs = pipeline_lr.named_steps['clf'].coef_[0]

top_spam = pd.Series(coefs, index=feature_names).nlargest(15)
top_ham  = pd.Series(coefs, index=feature_names).nsmallest(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_spam.plot(kind='barh', ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Top 15 termos — indicadores de SPAM')

top_ham.abs().sort_values().plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Top 15 termos — indicadores de HAM')

plt.tight_layout()
plt.show()

# 5. Word Embeddings <a id="5"></a>

O TF-IDF trata cada palavra como independente — não captura que "free" e "grátis" têm significados semelhantes. **Word Embeddings** representam palavras como vetores densos em um espaço de baixa dimensão, onde palavras semanticamente próximas ficam próximas no espaço vetorial.

## 5.1 Embeddings com Keras <a id="51"></a>

A camada `Embedding` do Keras aprende os vetores de palavras durante o treinamento, a partir dos próprios dados.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, GlobalMaxPooling1D, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow: {tf.__version__}')

In [ ]:
# parâmetros
VOCAB_SIZE   = 10000
MAX_LEN      = 100
EMBED_DIM    = 64

# tokenizando as mensagens
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

# padding — todas as sequências com o mesmo comprimento
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f'Vocabulário: {len(tokenizer.word_index)} palavras únicas')
print(f'Shape treino: {X_train_pad.shape} | Shape teste: {X_test_pad.shape}')

## 5.2 LSTM para Classificação de Texto <a id="52"></a>

Combinamos a camada `Embedding` com um **LSTM Bidirecional** — lê a sequência de frente para trás e de trás para frente, capturando contexto em ambas as direções.

In [ ]:
tf.random.set_seed(42)

modelo_lstm = Sequential([
    Embedding(VOCAB_SIZE, EMBED_DIM, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    GlobalMaxPooling1D(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

modelo_lstm.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

modelo_lstm.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

historico = modelo_lstm.fit(
    X_train_pad, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# curva de treinamento
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(historico.history['loss'],     label='Treino',    color='steelblue')
axes[0].plot(historico.history['val_loss'], label='Validação', color='coral')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(historico.history['accuracy'],     label='Treino',    color='steelblue')
axes[1].plot(historico.history['val_accuracy'], label='Validação', color='coral')
axes[1].set_title('Acurácia')
axes[1].legend()

plt.suptitle('LSTM Bidirecional — Curvas de Treinamento', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# avaliando no conjunto de teste
y_prob_lstm = modelo_lstm.predict(X_test_pad).flatten()
y_pred_lstm = (y_prob_lstm > 0.5).astype(int)

print('=== LSTM Bidirecional ===')
print(classification_report(y_test, y_pred_lstm, target_names=['Ham', 'Spam']))

# 6. BERT com HuggingFace <a id="6"></a>

**BERT (Bidirectional Encoder Representations from Transformers)** é um modelo pré-treinado em bilhões de textos que entende o contexto completo de cada palavra — "free" em "free time" tem significado diferente de "free prize". Em vez de treinar do zero, fazemos **fine-tuning**: adaptamos o BERT pré-treinado para nossa tarefa específica de detecção de spam.

![BERT](https://i.imgur.com/iMHxWKh.png)

In [ ]:
!pip install transformers -q

In [ ]:
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
import tensorflow as tf

# usando o DistilBERT — versão mais leve do BERT (40% menor, 97% da performance)
MODEL_NAME = 'distilbert-base-uncased'

tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f'Modelo: {MODEL_NAME}')
print(f'Vocabulário: {tokenizer_bert.vocab_size} tokens')

In [ ]:
# exemplo de tokenização do BERT
exemplo = "Free prize! Call now to claim your reward!"
tokens = tokenizer_bert.tokenize(exemplo)
print('Tokens BERT:', tokens)

encoding = tokenizer_bert(exemplo, return_tensors='tf')
print('Input IDs:', encoding['input_ids'].numpy())

In [ ]:
# tokenizando o dataset completo para o BERT
MAX_LEN_BERT = 64

# usando as mensagens limpas (sem stemming — BERT prefere texto mais natural)
X_train_list = X_train.tolist()
X_test_list  = X_test.tolist()

def tokenizar_bert(textos, tokenizer, max_len):
    return tokenizer(
        textos,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='tf'
    )

enc_train = tokenizar_bert(X_train_list, tokenizer_bert, MAX_LEN_BERT)
enc_test  = tokenizar_bert(X_test_list,  tokenizer_bert, MAX_LEN_BERT)

print(f'Input IDs shape: {enc_train["input_ids"].shape}')

In [ ]:
# carregando o DistilBERT para classificação binária
modelo_bert = TFAutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

modelo_bert.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

modelo_bert.summary()

In [ ]:
# fine-tuning com 3 épocas
historico_bert = modelo_bert.fit(
    {'input_ids': enc_train['input_ids'], 'attention_mask': enc_train['attention_mask']},
    y_train.values,
    validation_split=0.15,
    epochs=3,
    batch_size=16,
    verbose=1
)

In [ ]:
# avaliando o BERT no conjunto de teste
logits_bert = modelo_bert.predict(
    {'input_ids': enc_test['input_ids'], 'attention_mask': enc_test['attention_mask']}
).logits

y_pred_bert = np.argmax(logits_bert, axis=1)

print('=== DistilBERT Fine-Tuning ===')
print(classification_report(y_test, y_pred_bert, target_names=['Ham', 'Spam']))

# 7. Comparativo Final <a id="7"></a>

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

def metricas(y_true, y_pred, nome):
    return {
        'Modelo':    nome,
        'Acurácia':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall':    recall_score(y_true, y_pred),
        'F1-Score':  f1_score(y_true, y_pred),
    }

resultados_finais = [
    metricas(y_test, y_pred_svm,  'TF-IDF + SVM'),
    metricas(y_test, y_pred_lstm, 'BiLSTM + Embedding'),
    metricas(y_test, y_pred_bert, 'DistilBERT (fine-tuning)'),
]

df_final = pd.DataFrame(resultados_finais).set_index('Modelo').round(4)
df_final.sort_values('F1-Score', ascending=False)

In [ ]:
df_final.plot(
    kind='bar', figsize=(11, 5),
    colormap='Set2', edgecolor='white'
)
plt.title('Comparativo de Modelos NLP — Detecção de SMS Spam')
plt.ylabel('Score')
plt.xticks(rotation=10, ha='right')
plt.ylim(0.85, 1.01)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# testando com mensagens novas
mensagens_novas = [
    "Congratulations! You've won a FREE iPhone. Click here to claim your prize now!",
    "Hey, are you coming to the meeting tomorrow at 3pm?",
    "URGENT: Your account has been suspended. Verify now to avoid charges.",
    "Can you pick up some milk on your way home?",
    "Limited offer! Get 50% off on all plans. Call 0800-FREE now!"
]

# previsão com TF-IDF + SVM
msgs_clean = [preprocessar(limpar_texto(m)) for m in mensagens_novas]
preds_svm  = pipeline_svm.predict(msgs_clean)

print('=== Previsões TF-IDF + SVM ===')
for msg, pred in zip(mensagens_novas, preds_svm):
    rotulo = '🚨 SPAM' if pred == 1 else '✅ HAM '
    print(f'{rotulo} | {msg[:70]}...' if len(msg) > 70 else f'{rotulo} | {msg}')

# 8. Conclusão <a id="8"></a>

Nesta aula construímos um pipeline completo de NLP aplicado à detecção de SMS spam em Telecom:

| Técnica | O que representa | Pontos fortes |
|---|---|---|
| Limpeza e normalização | Remoção de ruído | Base para qualquer abordagem |
| Stopwords + Stemming | Redução de dimensionalidade | Melhora modelos clássicos |
| TF-IDF | Bag of Words ponderado | Rápido, interpretável, excelente baseline |
| Naive Bayes | Probabilístico, independência entre features | Muito eficiente em texto |
| SVM Linear | Margem máxima no espaço TF-IDF | Alta precisão em texto esparso |
| Embedding + BiLSTM | Vetores densos + contexto sequencial | Captura ordem e semântica |
| DistilBERT | Contexto bidirecional pré-treinado | Estado da arte, generalizável |

**Quando usar cada abordagem:**

- **TF-IDF + SVM/LR**: dados limitados, latência baixa necessária, explicabilidade importante
- **LSTM**: moderada quantidade de dados, padrões sequenciais relevantes
- **BERT/DistilBERT**: máxima performance, dados suficientes para fine-tuning, tempo de inferência não é crítico

Na próxima aula vamos fechar o ciclo com **MLOps Básico** — como colocar esses modelos em produção, monitorar drift e construir pipelines reprodutíveis.

---
**Prof. Dr. Ahirton Lopes** | [LinkedIn](https://linkedin.com/in/ahirtonlopes) | [GitHub](https://github.com/ahirtonlopes)